In [1]:
import os
import sys
import torch
import string
from PIL import Image
from huggingface_hub import HfApi, login
from transformers import AutoModel, AutoProcessor, pipeline

sys.path.append(os.path.abspath(os.path.join("..")))
from deploy.captcha_convolutionaltransformer_base.configuration_captcha import CaptchaConfig
from deploy.captcha_convolutionaltransformer_base.processing_captcha import CaptchaProcessor
from deploy.captcha_convolutionaltransformer_base.modeling_captcha import CaptchaConvolutionalTransformer
from src.models.convoluationaltransformer.convtrans_v1 import Captcha_Convolutional_Transformer_V1

# Deploy Captcha-ConvolutionalTransformer-Base

In [2]:
# Load your old weights
old_weights = torch.load(Captcha_Convolutional_Transformer_V1.SAVE_DIR / "v4.pth", map_location="cpu")

# Initialize the new HF-style model
config = CaptchaConfig()
hf_model = CaptchaConvolutionalTransformer(config)

# Load the weights into the HF-style model
hf_model.load_state_dict(old_weights)

# Save locally as a standard HF repo
hf_model.save_pretrained("../deploy/captcha_convolutionaltransformer_base")

/nfs/home/tpz8688/Captcha-Recognition/deploy/captcha_convolutionaltransformer_base/modeling_captcha.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [3]:
# Create the processor with your specific vocab
vocab = string.ascii_lowercase + string.ascii_uppercase + string.digits
processor = CaptchaProcessor(vocab=vocab)

# Save it into the SAME folder as your model
processor.save_pretrained("../deploy/captcha_convolutionaltransformer_base")

['../deploy/captcha_convolutionaltransformer_base/processor_config.json']

### Manually add to config.json

```
"auto_map": {
    "AutoConfig": "configuration_captcha.CaptchaConfig",
    "AutoModel": "modeling_captcha.CaptchaConvolutionalTransformer",
    "AutoProcessor": "processing_captcha.CaptchaProcessor"
  }
```

### Manually add to processor_config.json

```
"auto_map": {
    "AutoProcessor": "processing_captcha.CaptchaProcessor"
  }
```

### Manually add to config.json

```
"custom_pipelines": {
    "captcha-recognition": {
      "impl": "pipeline.CaptchaPipeline",
      "pt": ["AutoModel"],
      "type": "multimodal"
    }
  }
```

### Upload to HuggingFace

In [ ]:
api = HfApi()
# login(token="***")

# 1. Create the repository
repo_id = "Graf-J/captcha-conv-transformer-base"
api.create_repo(repo_id=repo_id, exist_ok=True)

# 2. Upload everything
api.upload_folder(
    folder_path="../deploy/captcha_convolutionaltransformer_base",
    repo_id=repo_id,
    commit_message="Add Live-Demo to Readme"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Graf-J/captcha-conv-transformer-base/commit/1896f25517e3e9c2905db37863bc18e774759646', commit_message='Add Live-Demo to Readme', commit_description='', oid='1896f25517e3e9c2905db37863bc18e774759646', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Graf-J/captcha-conv-transformer-base', endpoint='https://huggingface.co', repo_type='model', repo_id='Graf-J/captcha-conv-transformer-base'), pr_revision=None, pr_num=None)

### Test Code

In [5]:
from transformers import pipeline
from PIL import Image

# Initialize the pipeline
pipe = pipeline(
    task="captcha-recognition", 
    model="Graf-J/captcha-conv-transformer-base", 
    trust_remote_code=True
)

# Load and predict
img = Image.open("Vb4cG.jpg")
result = pipe(img)
print(f"Decoded Text: {result['prediction']}")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

configuration_captcha.py:   0%|          | 0.00/568 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Graf-J/captcha-conv-transformer-base:
- configuration_captcha.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pipeline.py:   0%|          | 0.00/561 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Graf-J/captcha-conv-transformer-base:
- pipeline.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

Loading weights:   0%|          | 0/43 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

processing_captcha.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Graf-J/captcha-conv-transformer-base:
- processing_captcha.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Decoded Text: Vb4cG


In [6]:
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor

# Load Model & Custom Processor
repo_id = "Graf-J/captcha-conv-transformer-base"
processor = AutoProcessor.from_pretrained(repo_id, trust_remote_code=True)
model = AutoModel.from_pretrained(repo_id, trust_remote_code=True)

model.eval()

# Load and process image
img = Image.open("Vb4cG.jpg")
inputs = processor(img) 

# Inference
with torch.no_grad():
    outputs = model(inputs["pixel_values"])
    logits = outputs.logits

# Decode the prediction via CTC logic
prediction = processor.batch_decode(logits)[0]
print(f"Prediction: '{prediction}'")


/nfs/home/tpz8688/.cache/huggingface/modules/transformers_modules/Graf_hyphen_J/captcha_hyphen_conv_hyphen_transformer_hyphen_base/fa9821697aaf0263f3b16ffed5e936bbe7c1a0a5/modeling_captcha.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Loading weights:   0%|          | 0/43 [00:00<?, ?it/s]

Prediction: 'Vb4cG'
